# Solution 05 on Google Colab — VLM LoRA fine-tune, synthetic augmented (tuned upsampling)

Same recipe as solution 04 -- **Qwen/Qwen3-VL-8B-Instruct**, LoRA, real data + upsampling + heuristic injection + optional LLM-assisted injection -- with **one change**: the rarity-based upsampling multiplier is capped at **3x** instead of solution 04's **8x** (`--max_upsample_multiplier`, set below).

## Why this exists

Comparing solution 02 (real data only) and solution 04 (synthetic augmented) on the actual SHROOM test-set submissions revealed a consistent pattern across all 4 languages: solution 04 won `Cor` (label-agnostic span correlation) but **lost** `CorLbl` (per-label correlation) in every single language. Direct investigation of `val_predictions.jsonl` for both solutions (comparing predicted label-sets against gold, per example) found the mechanism: solution 04 predicts **2.5x more spurious labels per example** than solution 02 -- i.e. it guesses a hallucination category that isn't actually present on a given example much more often. `score_cor_lbl` in `participant_kit/scorer.py` computes Spearman correlation *separately per label present in gold ∪ prediction*, then unweighted-averages across labels -- so every spurious label adds a near-zero-correlation term that drags that example's score down, which is exactly why Cor went up (more real hallucinations found) while CorLbl went down (more wrong guesses about which category).

The most likely cause: solution 04's real run computed rarity multipliers of `OCR=7x, other=5x, miscounting=4x` -- training the model to expect these categories far more often than their true ~3.8%/3.9%/8.2% frequency in the data, making it more trigger-happy on them. This notebook tests the direct, single-parameter fix: cap that multiplier lower so rare classes still get *some* boost (preserving solution 04's recall win) without the over-triggering. Everything else -- the injection heuristics, OCR/LLM-assisted methods, base model, and training recipe -- is unchanged from solution 04.

**Before running anything:** `Runtime > Change runtime type > T4/L4/A100 GPU`.

**Assumption about your Drive layout:** same as solutions 02/04 -- `solutions/`, `participant_kit/`, `data/`, `shroom-vis-images/` all live together under one folder on your Drive, and **solution 04 has already been run at least once** (this notebook reuses its `ocr_cache.json` and `llm_synth_cache.json` automatically if present, to avoid repeating the ~6hr EasyOCR pass and the ~3,756 API calls -- neither of those change with this fix, only the upsampling multiplier does; solution 03's `ocr_cache.json` is an equally valid fallback for the OCR step specifically, since that grounding logic is identical across all four synthetic-augmentation solutions -- but solution 03 never calls the LLM, so it has no `llm_synth_cache.json` to reuse). Set `DRIVE_PROJECT_PATH` below.

**Checkpointing, resume, sequence-length OOM protection**: identical to solution 04's notebook -- `CHECKPOINT_DIR` points at Drive, `--save_steps`/`--resume` protect against a session cutoff, and `train_lora.py`'s `--max_seq_len` (default 4096) skips pathologically long examples automatically. See solution 04's notebook for the full incident history behind these.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- Config: edit these ---
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SHROOM"   # <-- change to wherever your project actually lives on Drive
LOCAL_PROJECT_PATH = "/content/SHROOM"
GPU_INDEX = 0        # Colab only ever exposes one GPU -- leave this at 0
EPOCHS = 2
GRAD_ACCUM = 8
SAVE_STEPS = 250     # full-run checkpoint frequency (optimizer steps)
MAX_UPSAMPLE_MULTIPLIER = 3   # <-- the actual fix. Solution 04 used 8 (computed OCR=7x/other=5x/
                               # miscounting=4x); this caps all three at 3x instead. Lower further
                               # (e.g. 2) for an even more conservative rebalance if needed.

sol05_local = f"{LOCAL_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned"

# Checkpoint lives on Drive (not local disk) so it survives a session being killed mid-training --
# see solution 04's notebook for the incident this protects against.
CHECKPOINT_DIR = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned/checkpoint"

SKIP_LLM = True   # matches solution 03's default choice (not solution 04's) -- flip to False only
                   # if you want method (c) included here too. No evidence the LLM-assisted method
                   # itself helps; this fix targets method (a) specifically, not (c).
if not SKIP_LLM:
    os.environ["ANTHROPIC_API_KEY"] = ""   # paste a FRESH key here if SKIP_LLM=False. Never
                                            # share/commit this notebook afterward with a real key
                                            # still in this cell -- clear it back to "" first.

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("DRIVE_PROJECT_PATH:", DRIVE_PROJECT_PATH)
print("LOCAL_PROJECT_PATH:", LOCAL_PROJECT_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("MAX_UPSAMPLE_MULTIPLIER:", MAX_UPSAMPLE_MULTIPLIER)
print("SKIP_LLM:", SKIP_LLM)

In [ ]:
# Copy just the subfolders the pipeline actually needs from Drive to local disk.
# Skips anything already copied (and non-empty), so re-running this cell after a runtime
# restart is cheap. Fails loudly (instead of silently) if a source folder is missing on
# Drive or a copy comes out empty.
os.makedirs(LOCAL_PROJECT_PATH, exist_ok=True)
for sub in ["data", "shroom-vis-images", "participant_kit", "solutions"]:
    src = f"{DRIVE_PROJECT_PATH}/{sub}"
    dst = f"{LOCAL_PROJECT_PATH}/{sub}"
    if os.path.exists(dst) and os.listdir(dst):
        print(f"skip (already present locally, {len(os.listdir(dst))} entries): {sub}")
        continue
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"{src} does not exist on your Drive. Check DRIVE_PROJECT_PATH and the exact folder "
            f"name/casing on your Drive (run `!ls \"{DRIVE_PROJECT_PATH}\"` in a new cell to see "
            f"what's actually there)."
        )
    print(f"copying {sub} from Drive ... (this can take a few minutes for shroom-vis-images/)")
    !cp -r "{src}" "{dst}"
    n = len(os.listdir(dst)) if os.path.exists(dst) else 0
    if n == 0:
        raise RuntimeError(
            f"Copy of {sub} produced an empty directory at {dst} -- something went wrong "
            f"(check the cp output above for errors, e.g. permission issues or a Drive timeout)."
        )
    print(f"  -> {n} entries copied")
print("done")
!ls "{LOCAL_PROJECT_PATH}"

In [ ]:
!pip install -q "git+https://github.com/huggingface/transformers"
!pip install -q "huggingface-hub<1.0" "accelerate>=0.34" "peft>=0.13" sentencepiece
!pip install -q bitsandbytes anthropic easyocr sacremoses

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) < (2, 6):
    print("\ntorch is older than 2.6 -- upgrading (see solution 02's notebook for why):")
    !pip install -q -U torch torchvision torchaudio
    print("IMPORTANT: Runtime > Restart runtime now, then re-run this notebook from the top.")
else:
    print("torch version OK.")

In [ ]:
%cd {LOCAL_PROJECT_PATH}
!python -m solutions.common.scorer_local

### Reuse solution 04's caches

The OCR-grounding and LLM-assisted-injection logic (methods b and c) are unchanged from solution 04 -- only the upsampling multiplier (method a) differs. So instead of rebuilding the ~6hr EasyOCR cache or re-spending API calls, these cells check solution 05's own Drive path first, then fall back to reusing solution 04's existing cache (the real, already-executed r_max=8 + LLM run).

In [ ]:
# OCR cache: prefer solution 05's own (if this isn't the first run), else reuse solution 04's
# (the real executed run), else solution 03's (an equally valid OCR-only fallback -- same
# grounding logic, though it was never run with the LLM step).
import shutil

ocr_cache_drive_05 = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned/ocr_cache.json"
ocr_cache_drive_04 = f"{DRIVE_PROJECT_PATH}/solutions/04_vlm_lora_synthetic_augmented_llm/ocr_cache.json"
ocr_cache_drive_03 = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented/ocr_cache.json"
ocr_cache_local = f"{sol05_local}/ocr_cache.json"

SKIP_OCR_EXTRACT = False
os.makedirs(sol05_local, exist_ok=True)
if os.path.exists(ocr_cache_drive_05):
    print("Found solution 05's own ocr_cache.json on Drive -- reusing it.")
    shutil.copy(ocr_cache_drive_05, ocr_cache_local)
    SKIP_OCR_EXTRACT = True
elif os.path.exists(ocr_cache_drive_04):
    print("No solution 05 OCR cache yet -- reusing solution 04's existing ocr_cache.json (same "
          "grounding logic, unaffected by this fix).")
    shutil.copy(ocr_cache_drive_04, ocr_cache_local)
    SKIP_OCR_EXTRACT = True
elif os.path.exists(ocr_cache_drive_03):
    print("No solution 05/04 OCR cache yet -- reusing solution 03's existing ocr_cache.json "
          "(identical OCR step).")
    shutil.copy(ocr_cache_drive_03, ocr_cache_local)
    SKIP_OCR_EXTRACT = True
else:
    print("No existing OCR cache found for any solution on Drive -- this run will build one "
          "from scratch (slow, budget hours).")
print("SKIP_OCR_EXTRACT =", SKIP_OCR_EXTRACT)

In [ ]:
# LLM-assisted injection cache: same reuse pattern, only matters if SKIP_LLM=False above.
# Only solution 04 can have one -- solution 03 always skips method (c), so it never produces
# an llm_synth_cache.json.
llm_cache_drive_05 = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned/llm_synth_cache.json"
llm_cache_drive_04 = f"{DRIVE_PROJECT_PATH}/solutions/04_vlm_lora_synthetic_augmented_llm/llm_synth_cache.json"
llm_cache_local = f"{sol05_local}/llm_synth_cache.json"

if not SKIP_LLM:
    if os.path.exists(llm_cache_drive_05):
        print("Found solution 05's own llm_synth_cache.json on Drive -- reusing it.")
        shutil.copy(llm_cache_drive_05, llm_cache_local)
    elif os.path.exists(llm_cache_drive_04):
        print("Reusing solution 04's existing llm_synth_cache.json -- avoids re-spending the "
              "~3,756 API calls that run made (method (c) itself isn't changed by this fix).")
        shutil.copy(llm_cache_drive_04, llm_cache_local)
    else:
        print("No existing LLM cache found for either solution -- will call the API fresh.")
else:
    print("SKIP_LLM=True -- nothing to reuse here.")

Smoke test next. Uses a local-only checkpoint dir (not `CHECKPOINT_DIR`/Drive) so it can never overwrite real full-run progress, same as solution 04's notebook. Deliberately does not pass `--skip_ocr_extract` unless a cache was already found above.

In [ ]:
%cd {sol05_local}
skip_llm_flag = "--skip_llm" if SKIP_LLM else ""
skip_ocr_flag = "--skip_ocr_extract" if SKIP_OCR_EXTRACT else "

In [ ]:
!python run_pipeline.py --smoke_test --gpu_index {GPU_INDEX} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_llm_flag} {skip_ocr_flag}

In [ ]:
# Same cleanup as solution 04's notebook: if the smoke test ran real EasyOCR (SKIP_OCR_EXTRACT
# was False), it just wrote a 15-image-only ocr_cache.json locally. Clear it before the full run
# so that one rebuilds a real cache instead of silently reusing the tiny smoke-test one.
if not SKIP_OCR_EXTRACT and os.path.exists(ocr_cache_local):
    os.remove(ocr_cache_local)
    print("Removed the smoke-test-scoped ocr_cache.json -- the full run below will build/reuse a real one.")
else:
    print("Nothing to clean up (either no cache was built yet, or a real one is already in place).")

Once the smoke test finishes cleanly, run the full pipeline. Same checkpointing/resume/
sequence-length protections as solution 04's notebook -- see its notebook for the full incident
history. `--resume` is safe to always pass (a no-op on a fresh/empty `CHECKPOINT_DIR`), so
re-running this exact cell after a session cutoff just continues from the last save.

OOM fallback for a smaller GPU (T4/L4 instead of A100), same as solutions 02/04:
```
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_llm_flag} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume --max_pixels 401408 --target_modules q_proj,k_proj,v_proj,o_proj
```

In [ ]:
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} --max_upsample_multiplier {MAX_UPSAMPLE_MULTIPLIER} {skip_llm_flag} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume

In [ ]:
# Save the OCR cache back to Drive (under solution 05's own path) right away, in case it was
# built fresh this run.
if os.path.exists(ocr_cache_local):
    os.makedirs(os.path.dirname(ocr_cache_drive_05), exist_ok=True)
    !cp "{ocr_cache_local}" "{ocr_cache_drive_05}"
    print("OCR cache saved to Drive (solution 05's own path).")
else:
    print("No local ocr_cache.json found (unexpected) -- nothing to save.")

if not SKIP_LLM and os.path.exists(llm_cache_local):
    os.makedirs(os.path.dirname(llm_cache_drive_05), exist_ok=True)
    !cp "{llm_cache_local}" "{llm_cache_drive_05}"
    print("LLM-assisted injection cache saved to Drive (solution 05's own path).")

In [ ]:
# Copy the rest of the results back to Drive: predictions/scores, plus the augmented training set
# for provenance/debugging. checkpoint/ is NOT copied here -- the full-run cell above already saved
# it directly to CHECKPOINT_DIR on Drive throughout training.
dest = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned"
os.makedirs(dest, exist_ok=True)
for item in ["val_predictions.jsonl", "val_scores.json", "train_augmented.jsonl"]:
    src = f"{sol05_local}/{item}"
    if os.path.exists(src):
        print(f"copying {item} -> Drive ...")
        !cp -r "{src}" "{dest}/"
    else:
        print(f"skip (not found): {item}")
print("done -- check", dest, "in your Drive")

In [ ]:
import json
with open(f"{sol05_local}/val_scores.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Compare against solutions 02/04 before submitting anything

Before spending time on test-set prediction (multi-hour, see below), sanity-check the val numbers above against solution 04's real val results (`overall: iou=0.4532, cor=0.4880, cor_lbl=0.4113`) and solution 02's (`overall: iou=0.4380, cor=0.4712, cor_lbl=0.4172`). The goal of this fix is CorLbl recovering toward (or past) solution 02's 0.4172 while not fully giving back solution 04's Cor gain. If CorLbl didn't move, `MAX_UPSAMPLE_MULTIPLIER` may need to go lower still (e.g. 2) -- re-run the full pipeline with that value rather than proceeding to test prediction on a fix that didn't work.

## Test-set predictions (for SHROOM website submission)

Same as solutions 02/04: runs the trained checkpoint over the real, **unlabeled** `data/test.jsonl` (4,898 examples). No scoring step -- test.jsonl has no gold labels. Budget several hours for this (solution 04's own test run measured ~9.3s/example on an A100 -- roughly 12-13 hours for the full test set). `predict.py` has no periodic-save/resume support yet, so a session cutoff during this step loses the whole run -- ask if you want that added before running this on a long/unreliable session.

**Required before submitting**: `data/test.jsonl` mixes all 4 languages in one file, but the SHROOM format needs one file per language. The cells below split the output and run the official format checker to confirm it's actually submission-ready.

In [ ]:
test_predictions_path = f"{sol05_local}/test_predictions.jsonl"

predict_script = f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data/predict.py"
test_file_path = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"

!python "{predict_script}" --checkpoint "{CHECKPOINT_DIR}" --val_file "{test_file_path}" --output "{test_predictions_path}" --gpu_index {GPU_INDEX}

Split by language + run the official format checker (also validates id coverage and span bounds against `data/test.jsonl` as the reference, not just the one-file-per-language rule).

In [ ]:
split_script = f"{LOCAL_PROJECT_PATH}/solutions/common/split_predictions_by_language.py"
!python "{split_script}" "{test_predictions_path}"

In [ ]:
format_checker_script = f"{LOCAL_PROJECT_PATH}/participant_kit/format_checker.py"
reference_file = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"
split_files = " ".join(
    f'"{sol05_local}/test_predictions.{lang}.jsonl"' for lang in ["en", "fr", "it", "zh"]
)
!python "{format_checker_script}" --reference-file "{reference_file}" {split_files}

In [ ]:
# Copy the split, submission-ready files to Drive -- these are what you upload to the SHROOM site.
dest = f"{DRIVE_PROJECT_PATH}/solutions/05_vlm_lora_synthetic_tuned"
os.makedirs(dest, exist_ok=True)
for lang in ["en", "fr", "it", "zh"]:
    src = f"{sol05_local}/test_predictions.{lang}.jsonl"
    if os.path.exists(src):
        print(f"copying test_predictions.{lang}.jsonl -> Drive ...")
        !cp "{src}" "{dest}/"
    else:
        print(f"skip (not found): test_predictions.{lang}.jsonl")
print("done -- check", dest, "in your Drive")

In [ ]:
# Quick sanity peek: line count + first record, so you can eyeball the format before submitting.
import json

with open(f"{sol05_local}/test_predictions.en.jsonl") as f:
    lines = f.readlines()
print(f"{len(lines)} EN predictions written")
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))